human lung cancer

In [1]:
import pandas as pd
import scanpy as sc
from scipy import sparse
import numpy as np
import random
import os

import time
import torch
from memory_profiler import memory_usage
from datetime import datetime
from functools import wraps


def measure_resources(cuda_id: int = 0,task_des:str="task",save_dir:str="/home/cavin/jt/benchmark/experiments/results/run_metric/cluster_with_annotation"):
    """
    测量函数运行时的CPU内存、指定GPU的显存消耗(峰值)和运行时间。
    
    参数:
    cuda_id (int): 需要监控的 GPU 编号，例如传入 0 代表监控 cuda:0
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            
            device = None
            if torch.cuda.is_available():
                try:
                    if not torch.cuda.is_initialized():
                        torch.cuda.init()
                    device = torch.device(f"cuda:{cuda_id}")
                    torch.cuda.reset_peak_memory_stats(device)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 初始化监控失败: {e}")

            start_time = time.time()
            
            mem_usage_mb, result = memory_usage(
                (func, args, kwargs), 
                max_usage=True, 
                retval=True
            )
            
            end_time = time.time()
            execution_time = end_time - start_time
            
           
            mem_usage_gb = mem_usage_mb / 1024.0
            
            allocated_gb = 0.0
            cached_gb = 0.0

            
            if device is not None:
                try:
                    allocated_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                    cached_gb = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 显存统计失败: {e}")

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # ===========================
            # 美观的表格式输出
            # ===========================
            report = f"""
============================================================
RESOURCE USAGE REPORT: {func.__name__}
============================================================
Timestamp                   : {timestamp}
Target GPU ID               : {cuda_id}
Execution Time (Minutes)    : {execution_time/60:.4f}
Execution Time (Seconds)    : {execution_time:.2f}
CPU Peak Memory Usage (GB)  : {mem_usage_gb:.4f}
GPU Peak Allocated Mem (GB) : {allocated_gb:.4f}
GPU Peak Cached Mem (GB)    : {cached_gb:.4f}
CUDA Available              : {torch.cuda.is_available()}
============================================================
"""
            print(report)
            save_file_name = "record.csv"
            full_path = os.path.join(save_dir, save_file_name)
            os.makedirs(save_dir, exist_ok=True)
            new_record = {"Timestamp":timestamp,"Task":task_des,
                          "Target GPU ID ":cuda_id,
                          "Execution Time (Minutes)":execution_time / 60,
                          "Execution Time (Seconds)":execution_time,
                          "CPU Peak Memory Usage (GB)":mem_usage_gb,
                          "GPU Peak Allocated Mem (GB)":allocated_gb,
                          "GPU Peak Cached Mem (GB)":cached_gb}
            new_data_df = pd.DataFrame([new_record])
            if not os.path.isfile(full_path):
                # 如果文件不存在：直接正常写入（默认 mode='w'），自带表头
                new_data_df.to_csv(full_path, index=False)
                print(f"文件不存在，已创建新文件并写入表头：{full_path}")
            else:
                # 如果文件已存在：使用追加模式 (mode='a')，并且绝对不写表头 (header=False)
                new_data_df.to_csv(full_path, mode='a', header=False, index=False)
                print(f"文件已存在，数据已成功追加至末尾。")
            return result
        return wrapper
    return decorator



def main_gene_selection(X_df, gene_list):
    """
    Describe:
        rebuild the input adata to select target genes encode protein 
    Parameters:
        adata->`~anndata.AnnData` object: adata with var index_name by gene symbol
        gene_list->list: wanted target gene 
    Returns:
        adata_new->`~anndata.AnnData` object
        to_fill_columns->list: zero padding gene
    """
    to_fill_columns = list(set(gene_list) - set(X_df.columns))
    padding_df = pd.DataFrame(np.zeros((X_df.shape[0], len(to_fill_columns))), 
                              columns=to_fill_columns, 
                              index=X_df.index)
    X_df = pd.DataFrame(np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
                        index=X_df.index, 
                        columns=list(X_df.columns) + list(padding_df.columns))
    X_df = X_df[gene_list]
    
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    return X_df, to_fill_columns,var


In [2]:
simple_path = '/home/cavin/jt/benchmark/data/human_lung_cancer/SMI_Lung.h5ad'
adata = sc.read_h5ad(simple_path)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [3]:
# highly variable genes
N_HVG = 2000
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor='seurat_v3')
adata = adata[:, adata.var['highly_variable']]

In [4]:
# X_df= pd.DataFrame(sparse.csr_matrix.toarray(adata.X),index=adata.obs.index.tolist(),columns=adata.var.index.tolist()) # read from csv file
X_df= pd.DataFrame(adata.X.toarray(),index=adata.obs.index.tolist(),columns=adata.var.index.tolist()) # read from csv file
gene_list_df = pd.read_csv('/home/cavin/jt/benchmark/raw_code/scFoundation/scFoundation/OS_scRNA_gene_index.19264.tsv', header=0, delimiter='\t')
gene_list = list(gene_list_df['gene_name'])
X_df, to_fill_columns, var = main_gene_selection(X_df, gene_list)
adata_uni = sc.AnnData(X_df)
adata_uni.obs = adata.obs
adata_uni.uns = adata.uns


/home/cavin/anaconda3/envs/geneformer/lib/python3.10/site-packages/anndata/_core/anndata.py:787: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  getattr(self, attr).index = value


In [5]:
X_df.to_parquet("/home/cavin/jt/benchmark/experiments/middle_data/scfoundation_cosmx_human_lung_cancer_X_df.parquet")

In [6]:
gene_list_df = pd.read_csv('/home/cavin/jt/benchmark/raw_code/scFoundation/scFoundation/model/OS_scRNA_gene_index.19264.tsv', header=0, delimiter='\t')
gene_list = list(gene_list_df['gene_name'])
random.seed(0)
np.random.seed(0)  # numpy random generator
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
class Arg:
    task_name="lung_cancer"
    input_type="singlecell"
    output_type="cell"
    pool_type="all"
    data_path="/home/cavin/jt/benchmark/experiments/middle_data/scfoundation_cosmx_human_lung_cancer_X_df.parquet"
    save_path="/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations"
    pre_normalized="F"
    version="ce"
    model_path="/home/cavin/jt/benchmark/models/scFoundation/models.ckpt"
    tgthighres="t4"
    batch_size=16

In [8]:
args = Arg()
gexpr_feature=pd.read_parquet(args.data_path)
if gexpr_feature.shape[1]<19264:
        print('covert gene feature into 19264')
        gexpr_feature, to_fill_columns,var = main_gene_selection(gexpr_feature,gene_list)
        assert gexpr_feature.shape[1]>=19264
print(gexpr_feature.shape)

(87589, 19264)


In [9]:
import sys
from tqdm import tqdm
sys.path.insert(0, '/home/cavin/jt/benchmark/raw_code/scFoundation/scFoundation/model')
from load import load_model_frommmf,gatherData

@measure_resources(task_des="scFoundation run cosmx human lung cancer")
def get_emb():
    #Load model
    ckpt_path = args.model_path
    if args.output_type == 'cell':
        if args.version == 'ce':
            key = 'cell'
        elif args.version == 'rde':
            key = 'rde'
        else:
            raise ValueError('No version found')
    pretrainmodel,pretrainconfig = load_model_frommmf(ckpt_path,key)
    pretrainmodel.eval()

    geneexpemb=[]
    batchcontainer = []

    #Inference
    n = gexpr_feature.shape[0] // args.batch_size
    if gexpr_feature.shape[0] % args.batch_size != 0:
        n += 1
    for i in tqdm(range(n)):
        with torch.no_grad():
            #Bulk
            start_inx = i * args.batch_size
            end_inx = start_inx + args.batch_size
            if end_inx > gexpr_feature.shape[0]:
                end_inx = gexpr_feature.shape[0]
            if args.input_type == 'bulk':
                if args.pre_normalized == 'T':
                    totalcount = gexpr_feature.iloc[i,:].sum()
                elif args.pre_normalized == 'F':
                    totalcount = np.log10(gexpr_feature.iloc[i,:].sum())
                else:
                    raise ValueError('pre_normalized must be T or F')
                tmpdata = (gexpr_feature.iloc[i,:]).tolist()
                pretrain_gene_x = torch.tensor(tmpdata+[totalcount,totalcount]).unsqueeze(0).cuda()
                data_gene_ids = torch.arange(19266, device=pretrain_gene_x.device).repeat(pretrain_gene_x.shape[0], 1)
            
            #Single cell
            elif args.input_type == 'singlecell':
                # pre-Normalization
                if args.pre_normalized == 'F':
                    batch_arr = gexpr_feature.iloc[start_inx:end_inx, :].values
                    row_sums = batch_arr.sum(axis=1, keepdims=True)
                    row_sums[row_sums == 0] = 1
                    tmpdata = np.log1p(batch_arr / row_sums * 1e4).tolist()
                elif args.pre_normalized == 'T':
                    tmpdata = (gexpr_feature.iloc[start_inx:end_inx,:]).tolist()
                elif args.pre_normalized == 'A':
                    tmpdata = (gexpr_feature.iloc[start_inx:end_inx,:-1]).tolist()
                else:
                    raise ValueError('pre_normalized must be T,F or A')

                if args.pre_normalized == 'A':
                    totalcount = gexpr_feature.iloc[start_inx:end_inx,-1]
                else:
                    totalcount = gexpr_feature.iloc[start_inx:end_inx,:].sum(axis=1)

                # select resolution
                resolution = args.tgthighres
                if args.tgthighres[0] == 'f':
                    pretrain_gene_x = torch.tensor(tmpdata+[np.log10(totalcount*float(args.tgthighres[1:])),np.log10(totalcount)]).unsqueeze(0).cuda()
                elif args.tgthighres[0] == 'a':
                    pretrain_gene_x = torch.tensor(tmpdata+[np.log10(totalcount)+float(args.tgthighres[1:]),np.log10(totalcount)]).unsqueeze(0).cuda()
                elif args.tgthighres[0] == 't':
                    pretrain_gene_x = torch.cat((torch.tensor(tmpdata,dtype=torch.float32),torch.tensor([[float(args.tgthighres[1:])] * totalcount.shape[0], np.log10(totalcount)],dtype=torch.float32).T),dim=1).cuda()
                    # pretrain_gene_x = torch.tensor(tmpdata+[float(args.tgthighres[1:]),np.log10(totalcount)]).unsqueeze(0).cuda()
                else:
                    raise ValueError('tgthighres must be start with f, a or t')
                data_gene_ids = torch.arange(19266, device=pretrain_gene_x.device).repeat(pretrain_gene_x.shape[0], 1)

            value_labels = pretrain_gene_x > 0
            x, x_padding = gatherData(pretrain_gene_x, value_labels, pretrainconfig['pad_token_id'])

            #Cell embedding
            if args.output_type=='cell':
                position_gene_ids, _ = gatherData(data_gene_ids, value_labels, pretrainconfig['pad_token_id'])
                x = pretrainmodel.token_emb(torch.unsqueeze(x, 2).float(), output_weight = 0)
                position_emb = pretrainmodel.pos_emb(position_gene_ids)
                x += position_emb
                geneemb = pretrainmodel.encoder(x,x_padding)

                geneemb1 = geneemb[:,-1,:]
                geneemb2 = geneemb[:,-2,:]
                geneemb3, _ = torch.max(geneemb[:,:-2,:], dim=1)
                geneemb4 = torch.mean(geneemb[:,:-2,:], dim=1)
                if args.pool_type=='all':
                    geneembmerge = torch.concat([geneemb1,geneemb2,geneemb3,geneemb4],axis=1)
                elif args.pool_type=='max':
                    geneembmerge, _ = torch.max(geneemb, dim=1)
                else:
                    raise ValueError('pool_type must be all or max')
                geneexpemb.append(geneembmerge.detach().cpu().numpy())

            #Gene embedding
            elif args.output_type=='gene':
                pretrainmodel.to_final = None
                encoder_data, encoder_position_gene_ids, encoder_data_padding, encoder_labels, decoder_data, decoder_data_padding, new_data_raw, data_mask_labels, decoder_position_gene_ids = getEncoerDecoderData(pretrain_gene_x.float(),pretrain_gene_x.float(),pretrainconfig)
                out = pretrainmodel.forward(x=encoder_data, padding_label=encoder_data_padding,
                            encoder_position_gene_ids=encoder_position_gene_ids,
                            encoder_labels=encoder_labels,
                            decoder_data=decoder_data,
                            mask_gene_name=False,
                            mask_labels=None,
                            decoder_position_gene_ids=decoder_position_gene_ids,
                            decoder_data_padding_labels=decoder_data_padding,
                            )
                out = out[:,:19264,:].contiguous()
                geneexpemb.append(out.detach().cpu().numpy())

            #Gene batch embedding
            elif args.output_type=='gene_batch':
                batchcontainer.append(pretrain_gene_x.float())
                if len(batchcontainer)==gexpr_feature.shape[0]:
                    batchcontainer = torch.concat(batchcontainer,axis=0)
                else:
                    continue
                pretrainmodel.to_final = None
                encoder_data, encoder_position_gene_ids, encoder_data_padding, encoder_labels, decoder_data, decoder_data_padding, new_data_raw, data_mask_labels, decoder_position_gene_ids = getEncoerDecoderData(batchcontainer,batchcontainer,pretrainconfig)
                out = pretrainmodel.forward(x=encoder_data, padding_label=encoder_data_padding,
                            encoder_position_gene_ids=encoder_position_gene_ids,
                            encoder_labels=encoder_labels,
                            decoder_data=decoder_data,
                            mask_gene_name=False,
                            mask_labels=None,
                            decoder_position_gene_ids=decoder_position_gene_ids,
                            decoder_data_padding_labels=decoder_data_padding,
                            )
                geneexpemb = out[:,:19264,:].contiguous().detach().cpu().numpy()
            #Gene_expression
            elif args.output_type=='gene_expression':
                encoder_data, encoder_position_gene_ids, encoder_data_padding, encoder_labels, decoder_data, decoder_data_padding, new_data_raw, data_mask_labels, decoder_position_gene_ids = getEncoerDecoderData(pretrain_gene_x.float(),pretrain_gene_x.float(),pretrainconfig)
                out = pretrainmodel.forward(x=encoder_data, padding_label=encoder_data_padding,
                            encoder_position_gene_ids=encoder_position_gene_ids,
                            encoder_labels=encoder_labels,
                            decoder_data=decoder_data,
                            mask_gene_name=False,
                            mask_labels=None,
                            decoder_position_gene_ids=decoder_position_gene_ids,
                            decoder_data_padding_labels=decoder_data_padding,
                            )
                out = out[:,:19264].contiguous()
                geneexpemb.append(out.detach().cpu().numpy())                
            else:
                raise ValueError('output_type must be cell or gene or gene_batch or gene_expression')
    geneexpemb = np.squeeze(np.vstack(geneexpemb))
    print(geneexpemb.shape)
    return geneexpemb

In [10]:
emb = get_emb()

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma

100%|██████████| 5475/5475 [07:36<00:00, 12.00it/s]


(87589, 3072)

RESOURCE USAGE REPORT: get_emb
Timestamp                   : 2026-03-05 18:09:39
Target GPU ID               : 0
Execution Time (Minutes)    : 7.6377
Execution Time (Seconds)    : 458.26
CPU Peak Memory Usage (GB)  : 47.6607
GPU Peak Allocated Mem (GB) : 1.0505
GPU Peak Cached Mem (GB)    : 2.9238
CUDA Available              : True

文件已存在，数据已成功追加至末尾。


In [11]:
emb.shape

(87589, 3072)

In [12]:
emb_df = pd.DataFrame(
    emb, 
    index=adata.obs_names,
    columns=[f"scGPT_dim_{i}" for i in range(emb.shape[1])]
)
save_path = "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations/scfoundation_human_lung_cancer_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")

In [13]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [14]:
adata.obsm["X_scfoundation"] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial', 'X_scfoundation'